In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm
import warnings
import shutil
import time

# 自动尝试安装 EduData
try:
    from EduData import get_data
except ImportError:
    print("正在安装 EduData...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
    from EduData import get_data

warnings.filterwarnings('ignore')

# =========================================
# 1. 基础配置 (Configuration for Junyi)
# =========================================

class Config:
    """实验超参数配置 - 适配 Junyi 数据集"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.3

        # 训练参数
        self.batch_size = 128
        self.epochs = 10
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data/junyi"
        self.dataset_name = "junyi"

        # *** 关键设置 ***: 限制读取行数以防内存溢出 (Junyi 全量数据极大)
        self.max_rows = 500000

# =========================================
# 2. 数据处理模块 (Data Processing)
# =========================================

class JunyiDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config

    def _find_data_file(self):
        """在数据目录中搜索目标 CSV 文件"""
        if not os.path.exists(self.config.data_dir):
            return None

        for root, _, files in os.walk(self.config.data_dir):
            for file in files:
                # 寻找包含 problemlog 且后缀为 .csv 的文件
                if "problemlog" in file.lower() and file.lower().endswith(".csv"):
                    return os.path.join(root, file)
        return None

    def download_data(self):
        """核心逻辑：检查本地是否存在数据，若无则下载"""
        # 1. 首先检查本地是否已经有现成的文件，避免重复运行重新下载
        existing_file = self._find_data_file()
        if existing_file:
            print(f">>> 检测到本地已存在数据文件: {existing_file}，跳过下载流程。")
            return existing_file

        print(f">>> 本地未找到数据，准备下载数据集: {self.config.dataset_name}...")

        max_retries = 3
        for attempt in range(max_retries):
            try:
                os.makedirs(self.config.data_dir, exist_ok=True)
                print(f"尝试下载 (第 {attempt + 1}/{max_retries} 次)...")
                get_data(self.config.dataset_name, self.config.data_dir)

                # 下载并解压后再次尝试定位
                target_file = self._find_data_file()
                if target_file:
                    print(f"下载并定位成功: {target_file}")
                    return target_file

                break
            except Exception as e:
                print(f"下载或解压过程中出错: {e}")
                if attempt < max_retries - 1:
                    print("清理残留目录并准备重试...")
                    if os.path.exists(self.config.data_dir):
                        shutil.rmtree(self.config.data_dir)
                    time.sleep(5)
                else:
                    raise

        # 最终确认
        target_file = self._find_data_file()
        if target_file:
            return target_file
        else:
            raise FileNotFoundError(f"无法在 {self.config.data_dir} 中定位 problemlog CSV 文件。")

    def process(self):
        file_path = self.download_data()
        print(f"加载数据中 (nrows={self.config.max_rows})...")

        # 读取数据
        df = pd.read_csv(file_path, nrows=self.config.max_rows)

        # 智能映射列名 - 解决 Junyi 不同版本列名差异问题
        rename_map = {}
        found_targets = set()

        for col in df.columns:
            low_col = col.lower()
            target = None
            if low_col == 'user_id' or (not target and 'user' in low_col and 'id' in low_col):
                target = 'user_id'
            elif low_col == 'exercise' or low_col == 'problem_id' or (not target and 'item' in low_col and 'id' in low_col):
                target = 'item_id'
            elif low_col == 'skill_id' or (not target and 'topic' in low_col) or (not target and 'skill' in low_col):
                target = 'skill_id'
            elif low_col == 'correct' or (not target and 'is_correct' in low_col):
                target = 'correct'
            elif low_col == 'timestamp' or (not target and 'time' in low_col):
                target = 'timestamp'

            # 确保每类目标列只映射一次，防止列名重复
            if target and target not in found_targets:
                rename_map[col] = target
                found_targets.add(target)

        df.rename(columns=rename_map, inplace=True)

        # *** 关键回退逻辑 ***: Junyi 原始数据中 exercise 即为知识点，若无独立的 skill_id，则直接复制 item_id
        if 'skill_id' not in df.columns:
            if 'item_id' in df.columns:
                print(">>> 提示: 未检测到独立的 'skill_id' 列，自动将 'item_id' (Exercise) 作为知识点使用。")
                df['skill_id'] = df['item_id']
            else:
                raise KeyError(f"数据列映射失败。当前列名: {df.columns.tolist()}")

        # 最终检查基础必要列
        for req in ['user_id', 'item_id', 'skill_id', 'correct']:
            if req not in df.columns:
                raise KeyError(f"数据集中缺少必要列: {req}。当前识别列: {df.columns.tolist()}")

        # 数据清洗：转换正确性标志
        df['correct'] = df['correct'].apply(lambda x: 1 if str(x).lower() in ['1', '1.0', 'true', '1.0'] else 0)
        df.dropna(subset=['user_id', 'item_id', 'skill_id'], inplace=True)

        # 处理时间戳并排序 (防止由于列名映射导致的 DataFrame 类型列)
        if 'timestamp' in df.columns:
            if isinstance(df['timestamp'], pd.DataFrame):
                df['timestamp'] = df['timestamp'].iloc[:, 0]
            df.sort_values(by=['user_id', 'timestamp'], inplace=True)

        print("建立映射表 (Index Mapping)...")
        user_map = {val: i+1 for i, val in enumerate(df['user_id'].unique())}
        item_map = {val: i+1 for i, val in enumerate(df['item_id'].unique())}
        skill_map = {val: i+1 for i, val in enumerate(df['skill_id'].unique())}

        df['u_idx'] = df['user_id'].map(user_map)
        df['i_idx'] = df['item_id'].map(item_map)
        df['s_idx'] = df['skill_id'].map(skill_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(item_map) + 1
        self.config.num_concepts = len(skill_map) + 1

        # 统计题目难度特征
        item_difficulty = df.groupby('i_idx')['correct'].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_difficulty.items():
            diff_values[iid] = diff

        # 构造 Q-Matrix (题目与知识点关系矩阵)
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp = df[['i_idx', 's_idx']].drop_duplicates()
        for _, row in tmp.iterrows():
            q_k_relation[int(row['i_idx']), int(row['s_idx'])] = 1
        row_sum = q_k_relation.sum(axis=1, keepdims=True)
        q_k_relation = np.divide(q_k_relation, row_sum, out=np.zeros_like(q_k_relation), where=row_sum!=0)

        print("构造用户交互序列...")
        all_q, all_r, all_s = [], [], []
        for _, group in tqdm(df.groupby('u_idx')):
            if len(group) < 3: continue
            q_list = group['i_idx'].values
            r_list = group['correct'].values
            u_id = group['u_idx'].iloc[0]

            # 按 seq_len 切分长序列
            for i in range(0, len(q_list), self.config.seq_len):
                sub_q = q_list[i:i + self.config.seq_len]
                sub_r = r_list[i:i + self.config.seq_len]
                if len(sub_q) < 3: continue
                all_q.append(sub_q)
                all_r.append(sub_r)
                all_s.append(u_id)

        return all_q, all_r, all_s, q_k_relation, diff_values

# =========================================
# 3. 核心模型 HIG-TCAN (HeteroGraph + TCAN)
# =========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super().__init__()
        self.q_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.c_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('diff', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, q_ids, q_k_rel):
        k_weights = self.c_emb.weight
        q_k_agg = torch.matmul(q_k_rel, k_weights)
        d_feat = self.diff_proj(self.diff[q_ids].unsqueeze(-1))
        # 融合题目嵌入、知识点特征和难度特征
        return self.q_emb(q_ids) + F.embedding(q_ids, q_k_agg, padding_idx=0) + d_feat

class TCANBlock(nn.Module):
    def __init__(self, dim, k, dila, drop):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads=4, dropout=drop, batch_first=True)
        self.conv = nn.Conv1d(dim, dim, k, dilation=dila, padding=(k-1)*dila)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        res = x
        # 因果掩码注意力
        mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, need_weights=False, attn_mask=mask)
        # 因果卷积
        conv_in = attn_out.transpose(1, 2)
        conv_out = self.conv(conv_in)[:, :, :x.size(1)].transpose(1, 2)
        return self.drop(F.relu(self.norm(conv_out + res)))

class HIG_TCAN_Model(nn.Module):
    def __init__(self, config, q_k_rel, diffs):
        super().__init__()
        self.register_buffer('q_k_rel', torch.tensor(q_k_rel, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diffs)
        self.stu_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.proj = nn.Linear(config.embed_dim + 1, config.embed_dim)

        self.tcan = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])

        self.out = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q, r, s):
        q_feat = self.graph_emb(q, self.q_k_rel)
        x = self.proj(torch.cat([q_feat, r.unsqueeze(-1)], dim=-1))
        h = x
        for layer in self.tcan: h = layer(h)
        s_feat = self.stu_emb(s).unsqueeze(1).expand(-1, q.size(1), -1)
        return h, q_feat, s_feat

# =========================================
# 4. 实验运行与评估
# =========================================

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for q, r, m, s in loader:
            q, r, m, s = q.to(device), r.to(device), m.to(device), s.to(device)
            h, q_f, s_f = model(q, r, s)
            # 使用上一步的隐状态预测当前步的结果 (Shift 1)
            feat = torch.cat([h[:, :-1, :], q_f[:, 1:, :], s_f[:, 1:, :]], dim=-1)
            p = model.out(feat).squeeze(-1)

            mask = m[:, 1:].bool()
            if mask.sum() == 0: continue
            y_pred.append(p[mask].cpu().numpy())
            y_true.append(r[:, 1:][mask].cpu().numpy())

    y_true_all = np.concatenate(y_true) if y_true else np.array([])
    y_pred_all = np.concatenate(y_pred) if y_pred else np.array([])

    if len(y_true_all) == 0: return 0.5, 0.0
    return roc_auc_score(y_true_all, y_pred_all), accuracy_score(y_true_all, y_pred_all > 0.5)

def run():
    cfg = Config()
    proc = DataProcessor(cfg)

    try:
        q, r, s, qk, diff = proc.process()
    except Exception as e:
        print(f"数据处理流程出现错误: {e}")
        return

    # 划分训练集与验证集
    t_q, v_q, t_r, v_r, t_s, v_s = train_test_split(q, r, s, test_size=0.2, random_state=42)
    train_ld = DataLoader(JunyiDataset(t_q, t_r, t_s, cfg.seq_len), batch_size=cfg.batch_size, shuffle=True)
    val_ld = DataLoader(JunyiDataset(v_q, v_r, v_s, cfg.seq_len), batch_size=cfg.batch_size)

    model = HIG_TCAN_Model(cfg, qk, diff).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    print(f"\n开始训练 (设备: {cfg.device})...")
    for ep in range(cfg.epochs):
        model.train()
        l_sum = 0
        for q_b, r_b, m_b, s_b in tqdm(train_ld, desc=f"Epoch {ep+1}"):
            q_b, r_b, m_b, s_b = q_b.to(cfg.device), r_b.to(cfg.device), m_b.to(cfg.device), s_b.to(cfg.device)
            opt.zero_grad()
            h, q_f, s_f = model(q_b, r_b, s_b)
            # 训练阶段同样执行 Shift 预测
            feat = torch.cat([h[:, :-1, :], q_f[:, 1:, :], s_f[:, 1:, :]], dim=-1)
            p = model.out(feat).squeeze(-1)

            loss = F.binary_cross_entropy(p, r_b[:, 1:], reduction='none')
            loss = (loss * m_b[:, 1:]).sum() / (m_b[:, 1:].sum() + 1e-8)
            loss.backward()
            opt.step()
            l_sum += loss.item()

        auc, acc = evaluate(model, val_ld, cfg.device)
        print(f"Epoch {ep+1} | Loss: {l_sum/len(train_ld):.4f} | AUC: {auc:.4f} | ACC: {acc:.4f}")

if __name__ == "__main__":
    run()

downloader, INFO http://base.ustc.edu.cn/data/JunyiAcademy_Math_Practicing_Log/junyi.rar is saved as data/junyi/junyi.rar


>>> 本地未找到数据，准备下载数据集: junyi...
尝试下载 (第 1/3 次)...


downloader, INFO data/junyi/junyi.rar already exists. Send resume request after 157MB


downloader, INFO data/junyi/junyi.rar is unrar to data/junyi/junyi


下载并定位成功: ./data/junyi/junyi/junyi_ProblemLog_original.csv
加载数据中 (nrows=500000)...
建立映射表 (Index Mapping)...
构造用户交互序列...


...

Epoch 8 | Loss: 0.3765 | AUC: 0.7000 | ACC: 0.8317


Epoch 9: 100%|██████████| 235/235 [04:43<00:00,  1.21s/it]


Epoch 9 | Loss: 0.3675 | AUC: 0.6891 | ACC: 0.8229


Epoch 10: 100%|██████████| 235/235 [04:42<00:00,  1.20s/it]


Epoch 10 | Loss: 0.3578 | AUC: 0.6831 | ACC: 0.8218
